### Tiny Vision Transformer

  tiny vision transformer : for CIFAR-10 dataset
  
  CIFAR-10 dataset
  - 10개의 클래스
  - 32x32 픽셀 이미지
  - 10,000개의 이미지
  - 50,000개의 트레이닝 이미지
  - 10,000개의 테스트 이미지  
  
  Transformer Input 을 위해 이미지를 4x4 패치로 나눔
  - 총 8x8=64개의 패치로 나누어짐
  - 각 패치는 3채널 4x4 픽셀 이미지
  - 각 패치는 4x4x3=48 length vector

  ![CIFAR-10](https://github.com/ultralytics/docs/releases/download/0/cifar10-sample-image.avif)

  Data References:
    1. https://www.cs.toronto.edu/~kriz/cifar.html <br>
    2. https://developer-together.tistory.com/49 <br>
    3. https://tutorials.pytorch.kr/beginner/blitz/cifar10_tutorial.html?highlight=cifar

Packages import

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
# import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

print(torch.__version__)

2.2.2


Torch Device 

In [2]:
if torch.backends.mps.is_available():
  my_device = torch.device('mps')
elif torch.cuda.is_available():
  my_device = torch.device('cuda')
else:
  my_device = torch.device('cpu')

print(my_device)

mps


#### Image to Patch Embedding Vector for VIT

CNN + stride(with patch size) 를 이용하여 Image 를 슬라이딩하며 patch embedding vector 를 만듬

result dim : [batch size, patch size, embedding dim]

In [3]:
class PatchEmbedding(nn.Module):
  # patch_size = 4, in_channels = 3, embed_dim = 48 for CIFAR-10
  def __init__(self, patch_size=4, in_channels=3, embed_dim=48):
    super().__init__()

    # projection : 4x4 3채널 이미지를 1x1 48채널로 변환
    self.projection = nn.Conv2d(
      in_channels=in_channels,
      out_channels=embed_dim,
      kernel_size=patch_size,
      stride=patch_size,
    )

  def forward(self, x):
    # [B, 3, 32, 32] (CIFAR-10 기준, B는 배치 크기) > [B, 48, 8, 8]
    x = self.projection(x)
    # [B, 48, 8, 8] > [B, 48, 64] : [batch size, embedding dim, patch size]
    x = x.flatten(2) # start_dim=2, end_dim=-1


    """
      대부분의 트랜스포머 모델은 입력을 [batch_size, sequence_length, embedding_dim] 형식으로 받음
      따라서 이 형식으로 변환해주는 작업이 필요함
      transpose() : permute 함수와 달리 두 개의 차원만 맞교환 가능능
    """
    # [batch size, embedding dim, patch size] > [batch size, patch size, embedding dim]
    x = x.transpose(1, 2) 
    return x
    

#### class token, position embedding 을 학습 가능한 nn.Parameter 로 선언함.

**nn.Parameter** 

PyTorch에서 학습 가능한 파라미터(텐서)를 정의할 때 사용하는 도구입니다. 

일반 Tensor와는 달리, nn.Parameter는 모델의 파라미터로 자동 등록되어 학습 대상이 됩니다.



#### <font color="red">왜 tensor 가 아니라 학습 가능한 nn.Parameter 로 선언했을까 ?</font>


##### 1. Class Token (cls_token)
- 역할: 
  - 입력 이미지의 정보를 요약해서 최종적으로 **분류(classification)**에 사용되는 벡터입니다. <br>
- 이유: 
  - 고정된 상수로 두기보다는, 학습을 통해 분류 작업에 더 적합한 표현을 스스로 배우는 것이 훨씬 효과적입니다.
  - 초기에는 임의값으로 시작하지만, 학습이 진행되면서 점점 더 “이 이미지를 대표하는 요약 벡터” 역할을 잘 수행하게 됩니다.

##### 2. Position Embedding (pos_embedding)
  - 역할: 
    - 패치들의 **순서 정보(위치 정보)**를 Transformer에 전달합니다. <br>
  - 이유:
	  - 초기에는 sine/cosine 기반의 고정된 위치 임베딩을 사용할 수도 있지만, ViT에서는 일반적으로 학습 가능한 임베딩을 사용합니다.
	  - 실제 연구에서도 학습 가능한 위치 임베딩이 더 유연하고 성능이 더 좋을 수 있다고 보고되고 있습니다.


<br>

#### Dosovitskiy et al., 2020, “An Image is Worth 16x16 Words” 논문에서 학습 가능한 파라미터로 정의합니다.
   

In [4]:
class XD_TinyVit(nn.Module):
  # mlp_ratio : feed_forward block 의 hidden dim 과 embed_dim 의 비율
  # feed_forward hidden dim = embed_dim * mlp_ratio
  def __init__(self, image_size=32, patch_size=4, in_channels=3, num_classes=10, embed_dim=48, 
               num_heads=4, num_layers=4, dropout=0.1, mlp_ratio=4):
    super().__init__()

    # 이미지를 4x4 패치로 나누어진 패치의 개수
    num_patches = (image_size // patch_size) ** 2

    # 패치 임베딩 레이어 : 4x4 패치를 1x1 48채널로 변환
    self.patch_embedding = PatchEmbedding(patch_size, in_channels, embed_dim)

    # 클래스 토큰 : 패치 임베딩 레이어의 출력을 통해 클래스 토큰을 생성 
    self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
  
    # 위치 임베딩 : 클래스 패치와 패치 임베딩 레이어의 출력을 통해 위치 임베딩을 생성
    self.pos_embedding = nn.Parameter(torch.zeros(1, 1+num_patches, embed_dim))

    # torch transformer encoder 
    # first : torch transformer encoder layer 생성
    encoder_layer = nn.TransformerEncoderLayer(
      d_model=embed_dim,
      nhead=num_heads,
      dim_feedforward=int(embed_dim * mlp_ratio),
      dropout=dropout,
      activation='gelu',
      batch_first=True,
      norm_first=True
    )

    # second : torch transformer encoder 생성
    self.transformer_encoder = nn.TransformerEncoder(
      encoder_layer=encoder_layer,
      num_layers=num_layers
    )


    # last layer normalization
    self.last_lnorm = nn.LayerNorm(embed_dim)

    # classifier layer
    self.classifier = nn.Sequential(
      nn.Linear(embed_dim, embed_dim),
      nn.GELU(),
      nn.Dropout(dropout),
      nn.Linear(embed_dim, num_classes)
    )



  def forward(self, x):
    # patch embedding
    x = self.patch_embedding(x)

    # 클래스 토큰 추가
    cls_token = self.cls_token.expand(x.shape[0], -1, -1)

    # 패치 임베딩과 클래스 토큰 결합
    x = torch.cat([cls_token, x], dim=1)

    # 위치 임베딩 추가
    x = x + self.pos_embedding

    # transformer encoder
    x = self.transformer_encoder(x)

    # last layer normalization
    x = self.last_lnorm(x)

    # 클래스 토큰 추출
    cls_token = x[:, 0, :]

    # 분류 출력
    return self.classifier(cls_token)
  

### ViT model tranining based on CIFAR-10

#### Training variables

In [5]:
# training variables
learning_rate = 1e-3

# batch size
batch_size = 128

#### Training data loading

이미지를 딥러닝 모델에서 사용하는 torch tensor(normalized)로 변환하기 위해 <br>
torchvision.transforms.Compose를 이용하여 이미지 변환 전처리 과정을 하나로 구성한다.

In [6]:
import torch.utils


# 1. 학습용 데이터 변환 정의 
# data augmentation (optional) -> to tensor -> normalization
transform_train = transforms.Compose([
  transforms.RandomVerticalFlip(),
  transforms.RandomHorizontalFlip(),
  transforms.ToTensor(), # 이미지를 텐서로 변환 (0~255 -> 0~1)
  transforms.Normalize((0.5, 0.5, 0.5), # RGB 채널별 평균 
                       (0.5, 0.5, 0.5)) # RGB 채널별 표준편차
]
)

# 2. CIFAR-10 학습 데이터셋 다운로드 및 변환 적용 
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, 
                                        download=True, transform = transform_train)

# 3. torch Dataloader로 배치 처리
train_loader = torch.utils.data.DataLoader(
  trainset, batch_size=batch_size, shuffle=True, num_workers=2
) 


# 1. 검증용 데이터 변환 졍의
transform_test = transforms.Compose([
  transforms.ToTensor(), # 이미지를 텐서로 변환 (0~255 -> 0~1)
  transforms.Normalize((0.5, 0.5, 0.5), # RGB 채널별 평균 
                       (0.5, 0.5, 0.5)) # RGB 채널별 표준편차 
])


# 2. CIFAR-10 검증 데이터셋 다운로드 및 변환 적용 
testset = torchvision.datasets.CIFAR10(root='./data', train=False, 
                                        download=True, transform = transform_test)

# 3. torch Dataloader로 배치 처리 (not shuffle)
test_loader = torch.utils.data.DataLoader(
  testset, batch_size=batch_size, shuffle=False, num_workers=2
)

Files already downloaded and verified
Files already downloaded and verified


In [7]:
# vit model 생성 for CIFAR-10
model = XD_TinyVit(image_size=32, patch_size=4, in_channels=3,
                   num_classes=10, embed_dim=48, num_heads=4,
                   num_layers=4, dropout=0.1, mlp_ratio=4)
model.to(my_device)


# optimizer 생성
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# loss function
criterion = nn.CrossEntropyLoss()


/Users/hyunious/opt/anaconda3/envs/py312/lib/python3.12/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Training the Tiny ViT for CIRFAR-10 classification

In [9]:
epochs = 100

for epoch in range(epochs):
  # training mode
  model.train()
  # batch loss
  batch_loss = 0.

  # batch 단위로 학습을 수행한다.
  # batch(images, labels) 는 taining_loader 를 이용하여 로딩
  for i, (images, labels) in enumerate(train_loader):
    # images, labels to my device
    images, labels = images.to(my_device), labels.to(my_device)

    # forward pass
    outputs = model(images)

    # compute the loss
    loss = criterion(outputs, labels)
    batch_loss += loss.item()

    # zero the optimzer's gradient
    optimizer.zero_grad()
    # backward pass
    loss.backward()
    # update parameters
    optimizer.step()

  # epoch 단위로 average loss 출력
  print(f"Epoch[{epoch+1}] ",  f"loss: {batch_loss/(i+1):.4f}")


  # validation
  model.eval()
  correct = 0
  total = 0  # test 데이터 수를 얻기 위해 사용

  with torch.no_grad():
    for images, labels in test_loader:
      # images, labels to my device
      images, labels = images.to(my_device), labels.to(my_device)

      # forward pass : outputs.shape == [batch_size, 10] for CIFAR-10
      outputs = model(images)

      # image classification
      # torch.max : 최대값과 그 인덱스를 반환
      _, predicted = torch.max(outputs, dim=1)

      # correct output 
      correct += (predicted == labels).sum().item()

      total += labels.size(0)

  # epoch 단위로 average accuracy 출력
  accuracy = 100 * correct/total
  print(f"Test Accuracy: {accuracy:.2f}%")





ValueError: Invalid format specifier '.4f)' for object of type 'float'